## Convolutional Autoencoder for image colorization

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from skimage.color import rgb2lab, lab2rgb
import numpy as np
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
import glob
from PIL import Image

# ─────────────────────────────────────────
 1. DATASET
# ─────────────────────────────────────────

In [ ]:
#Load the Dataset CelebA 
class CelebAColorization(torch.utils.data.Dataset):
    """ def __init__(self, root, split='train', size=64):
        self.data = datasets.CelebA(
            root=root, split=split,
            transform=transforms.Compose([
                transforms.Resize((size, size)),
                transforms.ToTensor()  # gives [0,1] RGB
            ]),
            download=True
        )
        To dowload the dataset, you can use the previous code snippet. 
        Make sure to specify the correct path where you want to store the dataset.
        """
    
    
    
    def __init__(self, split='train', size=64):
        self.size = size
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor()
        ])
        
        # Get all image paths directly
        all_images = sorted(glob.glob("./Data/img_align_celeba/*.jpg"))
        
        # Split manually (CelebA standard splits)
        # 162770 train, 19867 val, 19962 test (will be replit into subset later for training, too big for CPU)
        if split == 'train':
            self.images = all_images[:162770]
        elif split == 'valid':
            self.images = all_images[162770:182637]
        else:
            self.images = all_images[182637:]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        img = self.transform(img)
        
        # Convert to numpy HxWxC in [0,1] 
        # Pytorch : [C,H,W] -> skimage want [H,W,C]
        img_np = img.permute(1, 2, 0).numpy()
        
        # Convert RGB → LAB
        img_lab = rgb2lab(img_np).astype(np.float32)
        
        # L channel: input (normalize to [-1, 1])
        L = img_lab[:, :, 0] / 50.0 - 1.0          # L is in [0, 100]
        
        # AB channels: target (normalize to [-1, 1])
        AB = img_lab[:, :, 1:] / 128.0              # AB is in [-128, 128]
        
        L = torch.tensor(L).unsqueeze(0)             # [1, H, W]
        AB = torch.tensor(AB).permute(2, 0, 1)       # [2, H, W]
        
        return L, AB
    



In [ ]:
# Test data base 

# Créer le dataset
dataset = CelebAColorization(split='train', size=64)

# Prendre une image au hasard
idx = random.randint(0, len(dataset) - 1)
L, AB = dataset[idx]

# Recréer l'image LAB
L = (L.squeeze().numpy() + 1.0) * 50.0        # back to [0,100]
AB = AB.permute(1, 2, 0).numpy() * 128.0      # back to [-128,128]

lab = np.concatenate([L[:, :, np.newaxis], AB], axis=2)

# Convert LAB → RGB
rgb = lab2rgb(lab)

# Affichage
plt.imshow(rgb)
plt.title("Image reconstruite")
plt.axis("off")
plt.show()

# ─────────────────────────────────────────
 2. MODEL
# ─────────────────────────────────────────

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        # ENCODER — progressively extract features, reduce spatial size
        self.encoder = nn.Sequential( #  container that chains layers together in order
            # Block 1: 64x64 → 32x32
            nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            # Block 2: 32x32 → 16x16
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            # Block 3: 16x16 → 8x8
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Block 4: 8x8 → 4x4 (bottleneck)
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        
        # DECODER — upsample back to original spatial size, predict AB
        self.decoder = nn.Sequential(
            # Block 1: 4x4 → 8x8
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Block 2: 8x8 → 16x16
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            # Block 3: 16x16 → 32x32
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            # Block 4: 32x32 → 64x64, output AB channels
            nn.ConvTranspose2d(64, 2, kernel_size=4, stride=2, padding=1),
            nn.Tanh()  # output in [-1, 1], matches our AB normalization
        )

    def forward(self, x):
        features = self.encoder(x)
        ab_pred = self.decoder(features)
        return ab_pred


# ─────────────────────────────────────────
 3. TRAINING
# ─────────────────────────────────────────

In [ ]:
def train(model, dataloader, optimizer, criterion, device, epoch):
    model.train()
    total_loss = 0
    
    for L, AB in tqdm(dataloader, desc=f"Epoch {epoch}"):
        L, AB = L.to(device), AB.to(device)
        
        optimizer.zero_grad()
        AB_pred = model(L)
        loss = criterion(AB_pred, AB)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

In [ ]:
# --- Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = ConvAutoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

train_dataset = CelebAColorization(split='train', size=64)

# num_workers=0 to avoid freezing on Windows
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)

In [ ]:
# Small test - only 1000 images, 2 epochs
from torch.utils.data import Subset

small_dataset = Subset(train_dataset, range(1000))
train_loader  = DataLoader(small_dataset, batch_size=64, shuffle=False, num_workers=0)

train_losses = []

for epoch in range(1, 3):
    loss = train(model, train_loader, optimizer, criterion, device, epoch)
    print(f"Epoch {epoch} | Loss: {loss:.4f}")
    train_losses.append(loss)

In [ ]:
# Only 2000 images, 5 epochs (~15-20 min on CPU)
small_dataset = Subset(train_dataset, range(2000))
small_loader  = DataLoader(small_dataset, batch_size=32, shuffle=False, num_workers=0)

train_losses = []

for epoch in range(1, 6):
    loss = train(model, train_loader, optimizer, criterion, device, epoch)
    print(f"Epoch {epoch} | Loss: {loss:.4f}")
    train_losses.append(loss)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(range(1, 21), train_losses, marker='o', label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Evolution')
plt.legend()
plt.grid(True)
plt.savefig('loss_curve.png', dpi=150)
plt.show()

In [ ]:
# Training on all the dataset (only on GG Colab GPU)
train_losses = []

for epoch in range(1, 21):
    loss = train(model, train_loader, optimizer, criterion, device, epoch)
    print(f"Epoch {epoch} | Loss: {loss:.4f}")
    train_losses.append(loss)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(range(1, 21), train_losses, marker='o', label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Evolution')
plt.legend()
plt.grid(True)
plt.savefig('loss_curve.png', dpi=150)
plt.show()


In [ ]:
# Validation loss evaluation

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for L, AB in tqdm(dataloader, desc="Evaluating"):
            L, AB = L.to(device), AB.to(device)
            AB_pred = model(L)
            loss = criterion(AB_pred, AB)
            total_loss += loss.item()
    return total_loss / len(dataloader)

val_dataset = CelebAColorization(split='valid', size=64)

small_dataset = Subset(train_dataset, range(2000))
small_loader  = DataLoader(small_dataset, batch_size=32, shuffle=False, num_workers=0)
val_loss = evaluate(model, small_loader, criterion, device)
print(f"Training Loss : {train_losses[-1]:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Difference: {abs(train_losses[-1] - val_loss):.4f}")

# ─────────────────────────────────────────
 4. VISUALIZATION
# ─────────────────────────────────────────

In [ ]:
def visualize(model, dataset, device, n=5):
    model.eval()
    fig, axes = plt.subplots(3, n, figsize=(n * 3, 9))
    
    with torch.no_grad():
        for i in range(n):
            #L, AB_true = dataset[i] # Fixed images
            
            # Random Visualisation
            idx = random.randint(0, len(dataset) - 1)
            L, AB_true = dataset[idx]
            
            AB_pred = model(L.unsqueeze(0).to(device)).squeeze(0).cpu()
            
            # Denormalize
            L_np  = (L.squeeze().numpy() + 1.0) * 50.0
            AB_t  = AB_true.permute(1, 2, 0).numpy() * 128.0
            AB_p  = AB_pred.permute(1, 2, 0).numpy() * 128.0
            
            # Reconstruct LAB images
            lab_true = np.stack([L_np, AB_t[:,:,0], AB_t[:,:,1]], axis=2)
            lab_pred = np.stack([L_np, AB_p[:,:,0], AB_p[:,:,1]], axis=2)
            
            rgb_true = np.clip(lab2rgb(lab_true), 0, 1)
            rgb_pred = np.clip(lab2rgb(lab_pred), 0, 1)
            
            axes[0, i].imshow(L_np, cmap='gray')
            axes[0, i].set_title("Input (Gray)")
            axes[1, i].imshow(rgb_pred)
            axes[1, i].set_title("Predicted")
            axes[2, i].imshow(rgb_true)
            axes[2, i].set_title("Ground Truth")
            
            for ax in axes[:, i]: ax.axis('off')
    
    plt.tight_layout()
    #plt.savefig("colorization_results.png", dpi=150)
    plt.show()

val_dataset = CelebAColorization(split='valid', size=64)
visualize(model, val_dataset, device)